# MCP Layer

In [1]:
import os
import logging
import base64
import json
import xgboost as xgb
import pickle
import pandas as pd
import numpy as np
import requests

from pathlib import Path
from pydantic import BaseModel, Field, ConfigDict
from typing import List, Optional, Dict, Any, Literal
from math import radians, sin, cos, sqrt, asin

from pinecone import Pinecone
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()

# Load API key from .env file
load_dotenv()
client = OpenAI()
openai_api_key = os.getenv("OPENAI_API_KEY")

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index = pc.Index("orlando-zoning-index")

# Zoning Layer

### Pydantic Schemas for Input/Output

In [2]:
class ZoningQueryInput(BaseModel):
    """Input schema for zoning law queries"""
    query: str = Field(
        ..., 
        description="Natural language question about Orlando zoning laws",
        examples=["What are the setback requirements for R-1A zones?"]
    )
    top_k: int = Field(
        default=5, 
        description="Number of relevant document chunks to retrieve",
        ge=1,
        le=20
    )
    similarity_threshold: float = Field(
        default=0.7,
        description="Minimum similarity score for retrieved chunks (0-1)",
        ge=0.0,
        le=1.0
    )

class SourceCitation(BaseModel):
    """Individual source citation with metadata"""
    source_document: str = Field(..., description="Name of the source document")
    page_number: Optional[int] = Field(None, description="Page number in source document")
    chunk_id: str = Field(..., description="Unique identifier for this chunk")
    chunk_text: str = Field(..., description="Relevant text excerpt from the document")
    similarity_score: float = Field(..., description="Cosine similarity score (0-1)")

class ZoningQueryOutput(BaseModel):
    """Output schema for zoning law queries"""
    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "query": "What are setback requirements for R-1A?",
                "answer": "For R-1A residential zones, the minimum setback requirements are...",
                "sources": [
                    {
                        "source_document": "orlando_zoning_code.docx",
                        "page_number": 42,
                        "chunk_id": "chunk_0123",
                        "chunk_text": "Setback requirements for R-1A zones...",
                        "similarity_score": 0.89
                    }
                ],
                "chunks_retrieved": 5,
                "total_tokens_used": 450
            }
        }
    )
    
    query: str = Field(..., description="Original query for reference")
    answer: str = Field(..., description="Generated answer based on retrieved zoning law context")
    sources: List[SourceCitation] = Field(..., description="Source citations supporting the answer")
    chunks_retrieved: int = Field(..., description="Number of chunks retrieved from vector store")
    total_tokens_used: Optional[int] = Field(None, description="Total tokens used in generation")

### Tool Class

In [3]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ZoningLawTool:
    """
    MCP-compatible tool for querying Orlando zoning law knowledge base.
    
    This tool uses RAG (Retrieval-Augmented Generation) to answer questions
    about Orlando municipal zoning codes by:
    1. Retrieving relevant document chunks from Pinecone vector store
    2. Generating accurate answers using OpenAI based on retrieved context
    """
    
    # MCP metadata for LLM discovery
    name: str = "zoning_law_query"
    description: str = """
    Query the Orlando municipal zoning code knowledge base using natural language.
    
    Use this tool when you need information about:
    - Zoning classifications (R-1A, R-2, C-1, etc.)
    - Setback requirements and lot dimensions
    - Permitted and conditional uses for zones
    - Density and height restrictions
    - Parking requirements
    - Variance and special exception rules
    - Any other Orlando zoning regulations
    
    The tool retrieves relevant sections from official zoning documents and 
    provides accurate, citation-backed answers.
    """
    
    def __init__(
        self,
        pinecone_api_key: Optional[str] = None,
        openai_api_key: Optional[str] = None,
        index_name: str = "orlando-zoning-laws",
        embedding_model: str = "text-embedding-3-small",
        generation_model: str = "gpt-4o-mini",
        namespace: str = "__default__"
    ):
        """
        Initialize the Zoning Law Tool.
        
        Args:
            pinecone_api_key: Pinecone API key (reads from env if not provided)
            openai_api_key: OpenAI API key (reads from env if not provided)
            index_name: Name of the Pinecone index
            embedding_model: OpenAI embedding model to use
            generation_model: OpenAI chat model for answer generation
            namespace: Pinecone namespace for zoning documents
        """
        # Get API keys from environment if not provided
        self.pinecone_api_key = pinecone_api_key or os.getenv("PINECONE_API_KEY")
        self.openai_api_key = openai_api_key or os.getenv("OPENAI_API_KEY")
        
        if not self.pinecone_api_key:
            raise ValueError("Pinecone API key not found. Set PINECONE_API_KEY environment variable.")
        if not self.openai_api_key:
            raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY environment variable.")
        
        # Initialize clients
        logger.info("Initializing Pinecone client...")
        self.pc = Pinecone(api_key=self.pinecone_api_key)
        self.index = self.pc.Index(index_name)
        self.namespace = namespace
        
        logger.info("Initializing OpenAI client...")
        self.openai_client = OpenAI(api_key=self.openai_api_key)
        
        # Store model names
        self.embedding_model = embedding_model
        self.generation_model = generation_model
        
        logger.info(f"ZoningLawTool initialized successfully with index '{index_name}'")
    
    def _create_embedding(self, text: str) -> List[float]:
        """Create embedding vector for query text."""
        try:
            response = self.openai_client.embeddings.create(
                model=self.embedding_model,
                input=text
            )
            return response.data[0].embedding
        except Exception as e:
            logger.error(f"Error creating embedding: {e}")
            raise
    
    def _retrieve_chunks(
        self, 
            query: str, 
        top_k: int = 5,
        similarity_threshold: float = 0.7) -> List[Dict[str, Any]]:
        """
        Retrieve relevant chunks from Pinecone with adaptive threshold.
    
        Args:
            query: Search query
            top_k: Number of chunks to retrieve
            similarity_threshold: Minimum similarity score
            
        Returns:
            List of matching chunks with metadata
        """
        try:
            # Create query embedding
            query_embedding = self._create_embedding(query)
            
            # Search Pinecone
            results = self.index.query(
                vector=query_embedding,
                top_k=top_k,
                namespace=self.namespace,
                include_metadata=True
            )
            
            # Filter by similarity threshold and format results
            chunks = []
            for match in results.matches:
                if match.score >= similarity_threshold:
                    chunks.append({
                        "id": match.id,
                        "score": match.score,
                        "metadata": match.metadata,
                        "text": match.metadata.get("text", "")
                    })
            
            # ADAPTIVE FALLBACK: If no chunks found with high threshold, try lower
            if not chunks and similarity_threshold > 0.5:
                logger.warning(
                    f"No chunks found with threshold {similarity_threshold}. "
                    f"Retrying with threshold 0.5..."
                )
                fallback_threshold = 0.5
                for match in results.matches:
                    if match.score >= fallback_threshold:
                        chunks.append({
                            "id": match.id,
                            "score": match.score,
                            "metadata": match.metadata,
                            "text": match.metadata.get("text", "")
                        })
                
                if chunks:
                    logger.info(
                        f"Retrieved {len(chunks)} chunks with fallback threshold {fallback_threshold}"
                    )
            
            # Final logging
            if chunks:
                logger.info(f"Retrieved {len(chunks)} chunks (threshold: {similarity_threshold})")
            else:
                logger.warning(f"No relevant chunks found for query: '{query}'")
            
            return chunks
            
        except Exception as e:
            logger.error(f"Error retrieving chunks: {e}")
            raise
    
    def _generate_answer(
        self, 
        query: str, 
        chunks: List[Dict[str, Any]]
    ) -> tuple[str, int]:
        """
        Generate answer using OpenAI based on retrieved chunks.
        
        Args:
            query: Original user query
            chunks: Retrieved document chunks
            
        Returns:
            Tuple of (generated_answer, tokens_used)
        """
        if not chunks:
            return "No relevant information found in the Orlando zoning code database for this query.", 0
        
        # Build context from chunks
        context_parts = []
        for i, chunk in enumerate(chunks, 1):
            source = chunk["metadata"].get("source", "Unknown")
            page = chunk["metadata"].get("page", "N/A")
            text = chunk["text"]
            context_parts.append(
                f"[Source {i}: {source}, Page {page}]\n{text}"
            )
        
        context = "\n\n".join(context_parts)
        
        # Create prompt
        system_prompt = """You are an expert on Orlando, Florida zoning laws and regulations. 
        
        Your role is to provide accurate, detailed answers about zoning codes based ONLY on the 
        provided context from official Orlando municipal code documents.

        Guidelines:
        - Answer directly and clearly
        - Cite specific sections or requirements when mentioned in the context
        - If the context doesn't fully answer the question, acknowledge what information is available
        - Use professional but accessible language
        - Include relevant details like setback distances, lot sizes, permitted uses, etc.
        - Do NOT make up information not present in the context"""

        user_prompt = f"""Based on the following excerpts from Orlando zoning code documents, 
        please answer this question:

        QUESTION: {query}

        CONTEXT FROM ZONING DOCUMENTS:
        {context}

        ANSWER:"""

        try:
            response = self.openai_client.chat.completions.create(
                model=self.generation_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,  # Low temperature for factual responses
                max_tokens=800
            )
            
            answer = response.choices[0].message.content
            tokens_used = response.usage.total_tokens
            
            return answer, tokens_used
            
        except Exception as e:
            logger.error(f"Error generating answer: {e}")
            raise
    
    def __call__(
        self, 
        query: str, 
        top_k: int = 5,
        similarity_threshold: float = 0.7
    ) -> ZoningQueryOutput:
        """
        Main entry point for the tool. Execute a zoning law query.
        
        Args:
            query: Natural language question about zoning laws
            top_k: Number of document chunks to retrieve (1-20)
            similarity_threshold: Minimum similarity score (0-1)
            
        Returns:
            ZoningQueryOutput with answer and source citations
        """
        logger.info(f"Executing zoning query: '{query}'")
        
        try:
            # Retrieve relevant chunks
            chunks = self._retrieve_chunks(query, top_k, similarity_threshold)
            
            # Generate answer
            answer, tokens_used = self._generate_answer(query, chunks)
            
            # Format source citations
            sources = []
            for chunk in chunks:
                citation = SourceCitation(
                    source_document=chunk["metadata"].get("source", "Unknown"),
                    page_number=chunk["metadata"].get("page"),
                    chunk_id=chunk["id"],
                    chunk_text=chunk["text"][:300] + "..." if len(chunk["text"]) > 300 else chunk["text"],
                    similarity_score=round(chunk["score"], 3)
                )
                sources.append(citation)
            
            # Build output
            output = ZoningQueryOutput(
                query=query,
                answer=answer,
                sources=sources,
                chunks_retrieved=len(chunks),
                total_tokens_used=tokens_used
            )
            
            logger.info(f"Query completed successfully. Retrieved {len(chunks)} chunks, used {tokens_used} tokens")
            return output
            
        except Exception as e:
            logger.error(f"Error executing zoning query: {e}")
            raise
    
    def get_tool_schema(self) -> Dict[str, Any]:
        """
        Return JSON schema for MCP protocol compatibility.
        This allows LLMs to understand how to call this tool.
        """
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": ZoningQueryInput.model_json_schema(),
            "output_schema": ZoningQueryOutput.model_json_schema()
        }

In [4]:
# Initialize the tool
zoning_tool = ZoningLawTool(
    index_name="orlando-zoning-index",
    namespace="__default__"
)

INFO:__main__:Initializing Pinecone client...
INFO:__main__:Initializing OpenAI client...
INFO:__main__:ZoningLawTool initialized successfully with index 'orlando-zoning-index'


In [5]:
# Simple query
result = zoning_tool(
    query="What size tree requires a permit?"
)

print(f"Answer: {result.answer}\n")
print(f"Sources used: {result.chunks_retrieved}")
for i, source in enumerate(result.sources, 1):
    print(f"\n[{i}] {source.source_document} (Page {source.page_number})")
    print(f"    Similarity: {source.similarity_score}")
    print(f"    Text: {source.chunk_text[:150]}...")

INFO:__main__:Executing zoning query: 'What size tree requires a permit?'
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Retrieved 5 chunks with fallback threshold 0.5
INFO:__main__:Retrieved 5 chunks (threshold: 0.7)
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Query completed successfully. Retrieved 5 chunks, used 1081 tokens


Answer: A tree requires a permit for removal if it has a diameter at breast height (dbh) of 10 inches or larger. This requirement is specified in multiple sections of the Orlando zoning code, including Sec. 60.209 and Sec. 65.640. It is prohibited to remove such trees without first obtaining a Tree Removal Permit, unless exempted by section 163.045 of the Florida Statutes.

Sources used: 5

[1] Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx (Page None)
    Similarity: 0.671
    Text: Sec. 60.209. General Requirements.
(a)	Tree Removal Permit Required. Removal of non-exempt existing 10" dbh or larger trees shall be prohibited withou...

[2] Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx (Page None)
    Similarity: 0.669
    Text: 6F. TREE PERMITS...

[3] Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx (Page None)
    Similarity: 0.643
    Text: Sec. 65.640. Tree Removal Permit.
When a Tree Removal Permit is Required. Unless exempt by section 163.045, Florida Statutes, persons desiro

In [6]:
# Example 2: Query with custom parameters
result = zoning_tool(
    query="What are permitted uses in commercial zones?",
    top_k=10,
    similarity_threshold=0.6
)

INFO:__main__:Executing zoning query: 'What are permitted uses in commercial zones?'
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Retrieved 5 chunks (threshold: 0.6)
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Query completed successfully. Retrieved 5 chunks, used 1516 tokens


In [7]:
# Access the tool schema (for MCP registration)
schema = zoning_tool.get_tool_schema()
print(schema)

{'name': 'zoning_law_query', 'description': '\n    Query the Orlando municipal zoning code knowledge base using natural language.\n\n    Use this tool when you need information about:\n    - Zoning classifications (R-1A, R-2, C-1, etc.)\n    - Setback requirements and lot dimensions\n    - Permitted and conditional uses for zones\n    - Density and height restrictions\n    - Parking requirements\n    - Variance and special exception rules\n    - Any other Orlando zoning regulations\n\n    The tool retrieves relevant sections from official zoning documents and \n    provides accurate, citation-backed answers.\n    ', 'input_schema': {'description': 'Input schema for zoning law queries', 'properties': {'query': {'description': 'Natural language question about Orlando zoning laws', 'examples': ['What are the setback requirements for R-1A zones?'], 'title': 'Query', 'type': 'string'}, 'top_k': {'default': 5, 'description': 'Number of relevant document chunks to retrieve', 'maximum': 20, 'm

In [8]:
# Convert output to dict for logging or API responses
output_dict = result.model_dump()
print(output_dict)

{'query': 'What are permitted uses in commercial zones?', 'answer': 'In commercial zones, the permitted uses include a variety of non-residential activities. Based on the provided excerpts from the Orlando zoning code documents, here are the key points regarding permitted uses in commercial zones:\n\n1. **General Non-Residential Uses**: The documents indicate that commercial, industrial, and office uses are not subject to the same sensitive land use regulations that apply to hospitals, clinics, nursing homes, childcare, and school uses (Source 1).\n\n2. **Retail Uses**: All retail uses are required to front on a major thoroughfare (Source 3). This suggests that retail establishments must be easily accessible and visible from significant roadways.\n\n3. **Conditional Uses**: Some uses may require a Conditional Use Permit, especially if they are located within 300 feet of a residential zoning district (Source 2). This indicates that while certain uses are generally permitted, proximity t

# Vision Discovery Layer

### Pydantic Schemas for Input/Output AND Tool Class

In [9]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class PropertyDamageInput(BaseModel):
    """Input schema for property damage assessment"""
    image_path: Optional[str] = Field(
        None,
        description="Path to a new property damage image file for analysis"
    )
    search_query: Optional[str] = Field(
        None,
        description="Text query to search existing damage assessments (e.g., 'water damage in ceiling')"
    )
    mode: Literal["analyze", "search"] = Field(
        default="analyze",
        description="Mode: 'analyze' for new image analysis, 'search' for finding similar existing damage"
    )
    top_k: int = Field(
        default=3,
        description="Number of similar damage cases to retrieve (search mode only)",
        ge=1,
        le=10
    )

class PropertyDamageOutput(BaseModel):
    """Output schema for property damage assessment"""
    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "file_name": "damage_04.jpg",
                "damage_type": "Ceiling Water Stain",
                "affected_system": "plumbing",
                "location": "Master Bedroom",
                "severity": 7,
                "urgency": "immediate",
                "visible_area_affected": "medium",
                "estimated_repair_cost": "$800-$2000",
                "secondary_damage_risk": "mold growth, ceiling collapse",
                "maintenance_summary": "Active roof leak or plumbing failure likely."
            }
        }
    )
    
    file_name: str = Field(..., description="Name of the image file")
    damage_type: str = Field(..., description="Type of damage observed")
    affected_system: Literal["roofing", "plumbing", "hvac", "electrical", "structural", "cosmetic"] = Field(
        ..., 
        description="Building system affected by the damage"
    )
    location: str = Field(..., description="Likely location in property")
    severity: int = Field(
        ..., 
        description="Severity score from 1-10",
        ge=1,
        le=10
    )
    urgency: Literal["immediate", "short-term", "monitor"] = Field(
        ...,
        description="How urgently repairs are needed"
    )
    visible_area_affected: Literal["small", "medium", "large"] = Field(
        ...,
        description="Size of affected area"
    )
    estimated_repair_cost: str = Field(
        ...,
        description="Rough cost range (e.g., $500-$1500)"
    )
    secondary_damage_risk: str = Field(
        ...,
        description="Potential consequences if untreated"
    )
    maintenance_summary: str = Field(
        ...,
        description="Brief expert assessment and recommended action"
    )
    similarity_score: Optional[float] = Field(
        None,
        description="Similarity score if retrieved from search (0-1)"
    )


class VisionSearchResult(BaseModel):
    """Result for search mode containing multiple similar damage cases"""
    query: str = Field(..., description="Original search query")
    results: List[PropertyDamageOutput] = Field(..., description="List of similar damage cases")
    results_count: int = Field(..., description="Number of results returned")


class VisionTool:
    """
    MCP-compatible tool for property damage assessment using GPT-4o-mini vision
    and semantic search over existing damage cases.
    
    Two modes:
    1. Analyze Mode: Assess a new property image for damage
    2. Search Mode: Find similar damage cases from existing assessments
    """
    
    # MCP metadata for LLM discovery
    name: str = "property_damage_assessment"
    description: str = """
    Analyze property damage images or search existing damage assessments.
    
    Two modes available:
    
    ANALYZE MODE (default):
    - Provide an image_path to analyze a new property damage image
    - Returns detailed damage assessment including severity, cost, urgency
    - Identifies affected systems (roofing, plumbing, HVAC, etc.)
    
    SEARCH MODE:
    - Provide a search_query to find similar damage cases from database
    - Returns multiple matching cases with similarity scores
    - Useful for comparing damage patterns or finding precedents
    
    Examples:
    - Analyze: image_path="damage_photo.jpg", mode="analyze"
    - Search: search_query="water damage ceiling", mode="search", top_k=5
    """
    
    # Damage assessment schema for structured output
    _damage_schema = {
        "name": "property_damage_assessment",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "file_name": {"type": "string"},
                "damage_type": {"type": "string"},
                "affected_system": {
                    "type": "string",
                    "enum": ["roofing", "plumbing", "hvac", "electrical", "structural", "cosmetic"]
                },
                "location": {"type": "string"},
                "severity": {"type": "integer"},
                "urgency": {
                    "type": "string",
                    "enum": ["immediate", "short-term", "monitor"]
                },
                "visible_area_affected": {
                    "type": "string",
                    "enum": ["small", "medium", "large"]
                },
                "estimated_repair_cost": {"type": "string"},
                "secondary_damage_risk": {"type": "string"},
                "maintenance_summary": {"type": "string"}
            },
            "required": [
                "file_name", "damage_type", "affected_system", "location",
                "severity", "urgency", "visible_area_affected",
                "estimated_repair_cost", "secondary_damage_risk", "maintenance_summary"
            ],
            "additionalProperties": False
        }
    }
    
    def __init__(
        self,
        openai_api_key: Optional[str] = None,
        pinecone_api_key: Optional[str] = None,
        index_name: str = "orlando-zoning-index",
        namespace: str = "property-damage",
        embedding_model: str = "text-embedding-3-small",
        vision_model: str = "gpt-4o-mini",
        max_tokens: int = 500
    ):
        """
        Initialize the Vision Tool.
        
        Args:
            openai_api_key: OpenAI API key (reads from env if not provided)
            pinecone_api_key: Pinecone API key (reads from env if not provided)
            index_name: Name of the Pinecone index
            namespace: Pinecone namespace for property damage embeddings
            embedding_model: OpenAI embedding model
            vision_model: Vision model to use (default: gpt-4o-mini)
            max_tokens: Maximum tokens for vision response
        """
        # Get API keys from environment if not provided
        self.openai_api_key = openai_api_key or os.getenv("OPENAI_API_KEY")
        self.pinecone_api_key = pinecone_api_key or os.getenv("PINECONE_API_KEY")
        
        if not self.openai_api_key:
            raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY environment variable.")
        if not self.pinecone_api_key:
            raise ValueError("Pinecone API key not found. Set PINECONE_API_KEY environment variable.")
        
        # Initialize clients
        logger.info("Initializing OpenAI client for vision analysis...")
        self.openai_client = OpenAI(api_key=self.openai_api_key)
        
        logger.info("Initializing Pinecone client...")
        self.pc = Pinecone(api_key=self.pinecone_api_key)
        self.index = self.pc.Index(index_name)
        self.namespace = namespace
        
        # Store model settings
        self.embedding_model = embedding_model
        self.vision_model = vision_model
        self.max_tokens = max_tokens
        
        logger.info(f"VisionTool initialized successfully")
    
    def _encode_image(self, image_path: Path) -> str:
        """Read image file and return base64 encoded string."""
        try:
            with open(image_path, "rb") as f:
                return base64.b64encode(f.read()).decode("utf-8")
        except Exception as e:
            logger.error(f"Error encoding image: {e}")
            raise
    
    def _create_embedding(self, text: str) -> List[float]:
        """Create embedding vector for text query."""
        try:
            response = self.openai_client.embeddings.create(
                model=self.embedding_model,
                input=text
            )
            return response.data[0].embedding
        except Exception as e:
            logger.error(f"Error creating embedding: {e}")
            raise
    
    def _analyze_new_image(self, image_path: Path, file_name: Optional[str] = None) -> PropertyDamageOutput:
        """
        Analyze a new image using GPT-4o-mini vision.
        
        Args:
            image_path: Path to the image file
            file_name: Optional override for file name
            
        Returns:
            PropertyDamageOutput with damage assessment
        """
        try:
            # Encode image to base64
            base64_image = self._encode_image(image_path)
            
            # Use provided file name or extract from path
            img_file_name = file_name or image_path.name
            
            # Create vision prompt
            response = self.openai_client.chat.completions.create(
                model=self.vision_model,
                messages=[
                    {
                        "role": "system",
                        "content": """You are an expert property inspector assessing damage for real estate valuation.
                        
                        Analyze the image and provide a detailed damage assessment.
                        
                        Be specific about damage type, severity, and repair recommendations.
                        
                        Base severity scores on impact to property value:
                        - 1-3: Minor cosmetic issues
                        - 4-6: Moderate damage requiring attention
                        - 7-9: Significant damage affecting value
                        - 10: Critical structural/safety issue"""
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": f"Analyze this property damage image. The file name is: {img_file_name}"
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{base64_image}",
                                    "detail": "high"
                                }
                            }
                        ]
                    }
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": self._damage_schema
                },
                max_tokens=self.max_tokens
            )
            
            # Parse JSON response
            damage_data = json.loads(response.choices[0].message.content)
            output = PropertyDamageOutput(**damage_data)
            
            logger.info(f"Successfully analyzed new image: {img_file_name}")
            return output
            
        except Exception as e:
            logger.error(f"Error analyzing image: {e}")
            raise
    
    def _search_similar_damage(self, query: str, top_k: int = 3) -> VisionSearchResult:
        """
        Search for similar damage cases in Pinecone.
        
        Args:
            query: Text description of damage to search for
            top_k: Number of results to return
            
        Returns:
            VisionSearchResult with multiple matching cases
        """
        try:
            # Create query embedding
            query_embedding = self._create_embedding(query)
            
            # Search Pinecone
            results = self.index.query(
                vector=query_embedding,
                top_k=top_k,
                namespace=self.namespace,
                include_metadata=True
            )
            
            # Parse results into PropertyDamageOutput objects
            damage_cases = []
            for match in results.matches:
                metadata = match.metadata
                
                # Add similarity score to metadata
                damage_output = PropertyDamageOutput(
                    **metadata,
                    similarity_score=round(match.score, 3)
                )
                damage_cases.append(damage_output)
            
            logger.info(f"Found {len(damage_cases)} similar damage cases for query: '{query}'")
            
            return VisionSearchResult(
                query=query,
                results=damage_cases,
                results_count=len(damage_cases)
            )
            
        except Exception as e:
            logger.error(f"Error searching damage cases: {e}")
            raise
    
    def __call__(
        self,
        image_path: Optional[str] = None,
        search_query: Optional[str] = None,
        mode: Literal["analyze", "search"] = "analyze",
        top_k: int = 3
    ):
        """
        Main entry point for the tool.
        
        Args:
            image_path: Path to image (analyze mode)
            search_query: Text query (search mode)
            mode: "analyze" or "search"
            top_k: Number of results (search mode)
            
        Returns:
            PropertyDamageOutput (analyze mode) or VisionSearchResult (search mode)
        """
        logger.info(f"Executing VisionTool in '{mode}' mode")
        
        try:
            if mode == "analyze":
                # Analyze new image
                if not image_path:
                    raise ValueError("image_path is required for analyze mode")
                
                img_path = Path(image_path)
                if not img_path.exists():
                    raise FileNotFoundError(f"Image file not found: {image_path}")
                
                return self._analyze_new_image(img_path)
            
            elif mode == "search":
                # Search existing damage cases
                if not search_query:
                    raise ValueError("search_query is required for search mode")
                
                return self._search_similar_damage(search_query, top_k)
            
            else:
                raise ValueError(f"Invalid mode: {mode}. Must be 'analyze' or 'search'")
                
        except Exception as e:
            logger.error(f"Error executing VisionTool: {e}")
            raise
    
    def get_tool_schema(self) -> dict:
        """Return JSON schema for MCP protocol compatibility."""
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": PropertyDamageInput.model_json_schema(),
            "output_schema": {
                "oneOf": [
                    PropertyDamageOutput.model_json_schema(),
                    VisionSearchResult.model_json_schema()
                ]
            }
        }

In [10]:
# Initialize the vision tool
vision_tool = VisionTool()

# Simple damage assessment
result = vision_tool(image_path="../data/prop-damage-images-resized-512/property_images_r3_c5_processed_by_imagy.png")

print(f"Damage Type: {result.damage_type}")
print(f"Affected System: {result.affected_system}")
print(f"Severity: {result.severity}/10")
print(f"Urgency: {result.urgency}")
print(f"Location: {result.location}")
print(f"Estimated Cost: {result.estimated_repair_cost}")
print(f"\nSummary: {result.maintenance_summary}")
print(f"\nSecondary Risks: {result.secondary_damage_risk}")

INFO:__main__:Initializing OpenAI client for vision analysis...
INFO:__main__:Initializing Pinecone client...
INFO:__main__:VisionTool initialized successfully
INFO:__main__:Executing VisionTool in 'analyze' mode
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Successfully analyzed new image: property_images_r3_c5_processed_by_imagy.png


Damage Type: water stains
Affected System: cosmetic
Severity: 4/10
Urgency: short-term
Location: interior wall near window
Estimated Cost: $300 - $600

Summary: Recommend cleaning and repainting affected areas, and checking for potential leaks or moisture sources.

Secondary Risks: mold growth if not addressed


In [11]:
# Get full output as dict
output_dict = result.model_dump()
print(json.dumps(output_dict, indent=2))

# Access tool schema (for MCP registration)
schema = vision_tool.get_tool_schema()
print(json.dumps(schema, indent=2))

{
  "file_name": "property_images_r3_c5_processed_by_imagy.png",
  "damage_type": "water stains",
  "affected_system": "cosmetic",
  "location": "interior wall near window",
  "severity": 4,
  "urgency": "short-term",
  "visible_area_affected": "medium",
  "estimated_repair_cost": "$300 - $600",
  "secondary_damage_risk": "mold growth if not addressed",
  "maintenance_summary": "Recommend cleaning and repainting affected areas, and checking for potential leaks or moisture sources.",
  "similarity_score": null
}
{
  "name": "property_damage_assessment",
  "description": "\n    Analyze property damage images or search existing damage assessments.\n\n    Two modes available:\n\n    ANALYZE MODE (default):\n    - Provide an image_path to analyze a new property damage image\n    - Returns detailed damage assessment including severity, cost, urgency\n    - Identifies affected systems (roofing, plumbing, HVAC, etc.)\n\n    SEARCH MODE:\n    - Provide a search_query to find similar damage case

In [12]:
# Test VisionTool search mode
print("=== TESTING VISION SEARCH MODE ===")
search_results = vision_tool(
    search_query="water damage on ceiling",
    mode="search",
    top_k=5
)

print(f"Found {search_results.results_count} similar cases:\n")
for i, case in enumerate(search_results.results, 1):
    print(f"[{i}] {case.damage_type} - {case.location}")
    print(f"    File: {case.file_name}")  # ← Added this line
    print(f"    Similarity: {case.similarity_score}")
    print(f"    Severity: {case.severity}/10, Cost: {case.estimated_repair_cost}\n")

INFO:__main__:Executing VisionTool in 'search' mode


=== TESTING VISION SEARCH MODE ===


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Found 5 similar damage cases for query: 'water damage on ceiling'


Found 5 similar cases:

[1] Ceiling Water Stain - Bathroom
    File: property_images_r4_c4_processed_by_imagy.png
    Similarity: 0.691
    Severity: 6/10, Cost: $1,000-$3,000

[2] Water Stain - Interior Room
    File: property_images_r3_c5_processed_by_imagy.png
    Similarity: 0.596
    Severity: 4/10, Cost: $300-$800

[3] Ceiling Damage/Mold - Living Room
    File: property_images_r1_c3_processed_by_imagy.png
    Similarity: 0.587
    Severity: 8/10, Cost: $2000-$5000

[4] Mold Growth - Ceiling
    File: property_images_r2_c4_processed_by_imagy.png
    Similarity: 0.575
    Severity: 8/10, Cost: $1500-$3000

[5] Mold Growth - Ceiling
    File: property_images_r1_c2_processed_by_imagy.png
    Similarity: 0.57
    Severity: 8/10, Cost: $1500-$3000



# FMV XGBoost

### Pydantic Input/Output

In [13]:
class FMVPredictionInput(BaseModel):
    """Input schema for FMV prediction"""
    # Location
    latitude: float = Field(..., description="Property latitude")
    longitude: float = Field(..., description="Property longitude")
    
    # Property characteristics
    land_sqft: float = Field(..., description="Land square footage (LND_SQFOOT)")
    living_area: float = Field(..., description="Total living area (TOT_LVG_AREA)")
    age: int = Field(..., description="Age of property in years")
    structure_quality: float = Field(..., description="Quality score (1-5 scale)")
    special_features_value: float = Field(default=0, description="Special features value (SPEC_FEAT_VAL)")
    
    # Distances (in meters/feet - your model uses these)
    rail_dist: float = Field(..., description="Distance to nearest rail")
    ocean_dist: float = Field(..., description="Distance to ocean")
    water_dist: float = Field(..., description="Distance to water body")
    center_dist: float = Field(..., description="Distance to city center")
    subcenter_dist: float = Field(..., description="Distance to subcenter")
    highway_dist: float = Field(..., description="Distance to highway")
    
    # Temporal
    month_sold: int = Field(default=6, description="Month (1-12)", ge=1, le=12)
    avno60plus: int = Field(default=0, description="AVN060+ indicator")


class FMVPredictionOutput(BaseModel):
    """Output schema for FMV prediction"""
    predicted_fmv: float = Field(..., description="Predicted fair market value in USD")
    property_summary: Dict[str, Any] = Field(..., description="Input property characteristics")
    confidence_interval: Optional[Dict[str, float]] = Field(
        None, 
        description="Estimated confidence range (if available)"
    )

### Tool Class

In [14]:
# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class FMVTool:
    """MCP-compatible tool for Fair Market Value prediction"""

    name: str = "predict_fair_market_value"
    description: str = """
    Predict the fair market value of Orlando properties using XGBoost model.
    
    Requires property characteristics:
    - Location (lat/lon)
    - Size (land, living area)
    - Age and quality
    - Distances to amenities
    """
    
    def __init__(self):
        # Load XGBoost model
        self.model = xgb.XGBRegressor()
        self.model.load_model('../output/orlando_fmv_xgboost_model_v2.json')
    
        # Load feature list
        self.feature_names = pd.read_csv('../output/feature_list_v2.csv', header=None)[0].tolist()

        with open('../output/spatial_kmeans.pkl', 'rb') as f:
            self.kmeans = pickle.load(f)
        
        logger.info(f"FMVTool initialized with {len(self.feature_names)} features")

    def _engineer_features(self, df, kmeans_model):
        """
        Create derived features to improve model performance.
        MUST match training exactly!
        """
        df = df.copy()
    
        # --- Ratio Features ---
        df['lot_utilization'] = df['TOT_LVG_AREA'] / df['LND_SQFOOT'].clip(lower=1)
        df['area_per_age'] = df['TOT_LVG_AREA'] / (df['age'] + 1)
        df['spec_feat_ratio'] = df['SPEC_FEAT_VAL'] / (df['TOT_LVG_AREA'] * 200).clip(lower=1)
        df['land_value_density'] = df['LND_SQFOOT'] / (df['CNTR_DIST'] + 1)
        
        # --- Interaction Features ---
        df['quality_x_area'] = df['structure_quality'] * df['TOT_LVG_AREA']
        df['quality_x_age'] = df['structure_quality'] / (df['age'] + 1)
        
        # --- Distance Features (LOG ONLY — no raw) ---
        distance_cols = ['RAIL_DIST', 'OCEAN_DIST', 'WATER_DIST', 'CNTR_DIST', 'SUBCNTR_DI', 'HWY_DIST']
        for col in distance_cols:
            df[f'{col}_log'] = np.log1p(df[col])
        
        # Drop raw distance columns
        df = df.drop(columns=distance_cols)
        
        # Combined urban accessibility score
        df['urban_access'] = 1 / (df['CNTR_DIST_log'] + df['SUBCNTR_DI_log'])
        
        # Water proximity premium indicator
        df['near_water'] = (df['WATER_DIST_log'] < np.log1p(1000)).astype(int)
        
        # Highway convenience
        df['hwy_convenience'] = np.where((df['HWY_DIST_log'] > np.log1p(500)) & (df['HWY_DIST_log'] < np.log1p(5000)), 1, 0)
        
        # --- Cyclical Encoding for Month ---
        df['month_sin'] = np.sin(2 * np.pi * df['month_sold'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month_sold'] / 12)
        
        # --- Location Features ---
        lat_center = 28.5383
        lon_center = -81.3792
        df['dist_from_center'] = np.sqrt((df['LATITUDE'] - lat_center)**2 + (df['LONGITUDE'] - lon_center)**2)
        
        # Spatial clusters
        if kmeans_model is not None:
            df['spatial_cluster'] = kmeans_model.predict(df[['LATITUDE', 'LONGITUDE']])
        
        # --- Binary Features ---
        df['has_spec_feat'] = (df['SPEC_FEAT_VAL'] > 0).astype(int)
        
        # --- Non-linear Quality ---
        df['quality_sq'] = df['structure_quality'] ** 2

        return df
    
    def __call__(
        self,

        #Numeric features - use float for continuous, int for discrete

        # Location - float
        latitude: float,           # e.g., 28.55
        longitude: float,          # e.g., -81.35

        # Property characteristics - float
        land_sqft: float,          # e.g., 8500.0
        living_area: float,        # e.g., 2000.0
        SPEC_FEAT_VAL: float,  # e.g., 10000.0
        structure_quality: float,  # e.g., 4.0 (can be decimal like 3.5)

        # Age - int
        age: int,                  # e.g., 15 (whole years)

        # Distances - float (can be meters or feet, depends on your training data)
        rail_dist: float,          # e.g., 5000.0
        ocean_dist: float,         # e.g., 55000.0
        water_dist: float,         # e.g., 1500.0
        center_dist: float,        # e.g., 40000.0
        subcenter_dist: float,     # e.g., 8000.0
        highway_dist: float,       # e.g., 3000.0

        # Temporal - int
        month_sold: int,           # e.g., 6 (June = 6)
        avno60plus: int            # e.g., 0 or 1 (binary indicator)
        
        ) -> FMVPredictionOutput:
        """Predict FMV for a property"""
        
        # 1. Create DataFrame from inputs
        property_data = {
            'LATITUDE': latitude,
            'LONGITUDE': longitude,
            'LND_SQFOOT': land_sqft,
            'TOT_LVG_AREA': living_area,
            'age': age,
            'structure_quality': structure_quality,
            'SPEC_FEAT_VAL':SPEC_FEAT_VAL,
            'RAIL_DIST':rail_dist,
            'OCEAN_DIST':ocean_dist,
            'WATER_DIST':water_dist,
            'CNTR_DIST':center_dist,
            'SUBCNTR_DI':subcenter_dist,
            'HWY_DIST':highway_dist,
            'month_sold':month_sold,
            'avno60plus':avno60plus
        }
        
        df = pd.DataFrame([property_data])
        
        # 2. Engineer features (MUST match training!)
        df_engineered = self._engineer_features(df, self.kmeans)
        
        # 3. Select features in correct order
        X = df_engineered[self.feature_names]
        
        # 4. Predict (model outputs log-transformed value)
        pred_log = self.model.predict(X)[0]
        
        # 5. Inverse transform (expm1 because you used log1p)
        predicted_fmv = np.expm1(pred_log)
        
        # 6. Return structured output
        return FMVPredictionOutput(
            predicted_fmv=float(predicted_fmv),
            property_summary=property_data
        )

In [15]:
# Complete example with ALL required inputs
fmv_tool = FMVTool()

INFO:__main__:FMVTool initialized with 30 features


In [16]:
examples = [
    ("Luxury Lakefront", 28.5448, -81.3725, 12000, 3500, 5, 5.0, 50000, 2000, 70000, 200, 5000, 3000, 4000),
    ("Older Suburban", 28.5200, -81.2500, 6500, 1400, 45, 3.0, 0, 15000, 60000, 5000, 25000, 12000, 8000),
    ("New Starter", 28.4500, -81.3800, 5000, 1200, 2, 4.0, 5000, 20000, 75000, 3000, 35000, 5000, 2000),
    ("Fixer-Upper", 28.5600, -81.4500, 7000, 1600, 35, 2.0, 0, 10000, 80000, 8000, 30000, 15000, 5000),
    ("Mid-Range Family", 28.6000, -81.3500, 8000, 2200, 20, 3.5, 15000, 8000, 65000, 4000, 20000, 6000, 3500)
]

print("="*70)
print("ORLANDO PROPERTY FMV PREDICTIONS")
print("="*70)

for name, lat, lon, land, area, age, qual, spec, rail, ocean, water, center, sub, hwy in examples:
    result = fmv_tool(
        latitude=lat, longitude=lon, land_sqft=land, living_area=area,
        age=age, structure_quality=qual, SPEC_FEAT_VAL=spec,
        rail_dist=rail, ocean_dist=ocean, water_dist=water,
        center_dist=center, subcenter_dist=sub, highway_dist=hwy,
        month_sold=6, avno60plus=0
    )
    print(f"{name:20s} - ${result.predicted_fmv:>12,.2f}")

print("="*70)

ORLANDO PROPERTY FMV PREDICTIONS
Luxury Lakefront     - $  925,541.50
Older Suburban       - $  282,254.12
New Starter          - $  288,375.28
Fixer-Upper          - $  358,916.12
Mid-Range Family     - $  466,468.09


### Open Street Maps

### Pydantic Schemas Input/Output

In [17]:
# ============================================================================
# PYDANTIC SCHEMAS
# ============================================================================

class AmenityDetail(BaseModel):
    """Individual amenity with location details"""
    name: Optional[str] = Field(None, description="Name of the amenity")
    distance_miles: float = Field(..., description="Distance from property in miles")
    latitude: float = Field(..., description="Amenity latitude")
    longitude: float = Field(..., description="Amenity longitude")


class AmenityCategoryInfo(BaseModel):
    """Summary info for a category of amenities"""
    count_within_1mile: int = Field(..., description="Number of amenities within 1 mile")
    nearest: Optional[AmenityDetail] = Field(None, description="Nearest amenity in this category")


class WalkabilityInput(BaseModel):
    """Input schema for walkability assessment"""
    latitude: float = Field(..., description="Property latitude", ge=-90, le=90)
    longitude: float = Field(..., description="Property longitude", ge=-180, le=180)
    radius_miles: float = Field(default=1.0, description="Search radius in miles", ge=0.1, le=5.0)


class WalkabilityOutput(BaseModel):
    """Output schema for walkability assessment"""
    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "walkability_score": 85,
                "walkability_interpretation": "Very Walkable",
                "groceries": {
                    "count_within_1mile": 3,
                    "nearest": {
                        "name": "Publix",
                        "distance_miles": 0.3,
                        "latitude": 28.54,
                        "longitude": -81.38
                    }
                },
                "summary": "This property has excellent walkability with 3 grocery stores..."
            }
        }
    )
    
    walkability_score: int = Field(..., description="Walkability score (0-100)", ge=0, le=100)
    walkability_interpretation: str = Field(..., description="Text interpretation of score")
    
    groceries: AmenityCategoryInfo = Field(..., description="Grocery store accessibility")
    restaurants: AmenityCategoryInfo = Field(..., description="Restaurant accessibility")
    schools: AmenityCategoryInfo = Field(..., description="School accessibility")
    parks: AmenityCategoryInfo = Field(..., description="Parks and recreation")
    hospitals: AmenityCategoryInfo = Field(..., description="Healthcare facilities")
    transit: AmenityCategoryInfo = Field(..., description="Public transit access")
    cafes: AmenityCategoryInfo = Field(..., description="Cafe accessibility")
    
    summary: str = Field(..., description="Human-readable summary of walkability")


# ============================================================================
# WALKABILITY TOOL CLASS
# ============================================================================

class WalkabilityTool:
    """
    MCP-compatible tool for assessing property walkability using OpenStreetMap data.
    
    Queries OpenStreetMap Overpass API for nearby amenities and calculates
    a walkability score based on proximity and variety of essential services.
    """
    
    name: str = "assess_walkability"
    description: str = """
    Assess property walkability and nearby amenities using OpenStreetMap data.
    
    Returns walkability score (0-100) and details about nearby:
    - Grocery stores and supermarkets
    - Restaurants and dining
    - Schools and education
    - Parks and recreation
    - Healthcare facilities
    - Public transit (bus stops, train stations)
    - Cafes and coffee shops
    
    Useful for evaluating neighborhood livability and car-dependency.
    """
    
    def __init__(self, timeout: int = 25):
        """
        Initialize Walkability Tool.
        
        Args:
            timeout: API request timeout in seconds
        """
        self.overpass_url = "http://overpass-api.de/api/interpreter"
        self.timeout = timeout
        logger.info("WalkabilityTool initialized")
    
    def _haversine_distance(self, lat1: float, lon1: float, lat2: float, lon2: float) -> float:
        """Calculate distance between two points in miles using Haversine formula."""
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * asin(sqrt(a))
        
        miles = 3956 * c
        return miles
    
    def _query_osm_amenities(self, lat: float, lon: float, radius_meters: int = 1600) -> dict:
        """
        Query OpenStreetMap Overpass API for amenities.
        
        Queries for amenity, shop, leisure, and highway tags.
        """
        query = f"""
        [out:json][timeout:{self.timeout}];
        (
        node["amenity"](around:{radius_meters},{lat},{lon});
        way["amenity"](around:{radius_meters},{lat},{lon});
        node["shop"="supermarket"](around:{radius_meters},{lat},{lon});
        way["shop"="supermarket"](around:{radius_meters},{lat},{lon});
        node["shop"="convenience"](around:{radius_meters},{lat},{lon});
        way["shop"="convenience"](around:{radius_meters},{lat},{lon});
        node["leisure"](around:{radius_meters},{lat},{lon});
        way["leisure"](around:{radius_meters},{lat},{lon});
        node["highway"="bus_stop"](around:{radius_meters},{lat},{lon});
        );
        out center;
        """
        
        try:
            response = requests.post(
                self.overpass_url,
                data={"data": query},
                timeout=self.timeout
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            logger.error(f"OSM API error: {e}")
            raise
    
    def _get_element_location(self, element: dict) -> Optional[tuple]:
        """Get lat/lon for both nodes and ways."""
        if element['type'] == 'node':
            return (element.get('lat'), element.get('lon'))
        elif element['type'] == 'way' and 'center' in element:
            return (element['center'].get('lat'), element['center'].get('lon'))
        return None
    
    def _process_amenities(self, raw_data: dict, property_lat: float, property_lon: float) -> dict:
        """Transform OSM data into structured insights."""
        elements = raw_data.get('elements', [])
        
        # Initialize categories
        categories = {
            'groceries': [],
            'restaurants': [],
            'schools': [],
            'parks': [],
            'hospitals': [],
            'transit': [],
            'cafes': []
        }
        
        for element in elements:
            tags = element.get('tags', {})
            location = self._get_element_location(element)
            
            if not location:
                continue
            
            elem_lat, elem_lon = location
            distance = self._haversine_distance(property_lat, property_lon, elem_lat, elem_lon)
            
            amenity_info = {
                'name': tags.get('name'),
                'distance_miles': round(distance, 2),
                'latitude': elem_lat,
                'longitude': elem_lon
            }
            
            # Categorize by tags
            amenity_type = tags.get('amenity')
            shop_type = tags.get('shop')
            leisure_type = tags.get('leisure')
            highway_type = tags.get('highway')
            
            # Groceries
            if shop_type in ['supermarket', 'convenience'] or amenity_type == 'marketplace':
                categories['groceries'].append(amenity_info)
            
            # Restaurants
            elif amenity_type in ['restaurant', 'fast_food', 'food_court']:
                categories['restaurants'].append(amenity_info)
            
            # Cafes
            elif amenity_type in ['cafe', 'bar', 'pub']:
                categories['cafes'].append(amenity_info)
            
            # Schools
            elif amenity_type in ['school', 'kindergarten', 'college', 'university']:
                categories['schools'].append(amenity_info)
            
            # Parks
            elif leisure_type in ['park', 'garden', 'playground'] or amenity_type == 'park':
                categories['parks'].append(amenity_info)
            
            # Healthcare
            elif amenity_type in ['hospital', 'clinic', 'doctors', 'pharmacy']:
                categories['hospitals'].append(amenity_info)
            
            # Transit
            elif highway_type == 'bus_stop' or amenity_type in ['bus_station', 'ferry_terminal']:
                categories['transit'].append(amenity_info)
        
        # Format output with nearest and count
        result = {}
        for category, amenities in categories.items():
            # Sort by distance
            amenities.sort(key=lambda x: x['distance_miles'])
            
            # Count within 1 mile
            count_within_1mile = len([a for a in amenities if a['distance_miles'] <= 1.0])
            
            # Get nearest
            nearest = amenities[0] if amenities else None
            
            result[category] = {
                'count_within_1mile': count_within_1mile,
                'nearest': nearest
            }
        
        return result
    
    def _calculate_walkability_score(self, amenities_data: dict) -> int:
        """
        Calculate walkability score (0-100).
        
        Scoring breakdown:
        - Groceries: 25 points max
        - Restaurants/Dining: 25 points max
        - Parks/Recreation: 20 points max
        - Transit: 15 points max
        - Schools: 15 points max
        """
        score = 0
        
        # Grocery accessibility (max 25 points)
        grocery_count = amenities_data['groceries']['count_within_1mile']
        nearest_grocery = amenities_data['groceries']['nearest']
        
        if nearest_grocery:
            distance = nearest_grocery['distance_miles']
            if distance <= 0.25:
                score += 25
            elif distance <= 0.5:
                score += 20
            elif distance <= 0.75:
                score += 15
            elif distance <= 1.0:
                score += 10
        
        # Restaurant/Dining diversity (max 25 points)
        restaurant_count = amenities_data['restaurants']['count_within_1mile']
        cafe_count = amenities_data['cafes']['count_within_1mile']
        dining_total = restaurant_count + cafe_count
        
        if dining_total >= 20:
            score += 25
        elif dining_total >= 10:
            score += 20
        elif dining_total >= 5:
            score += 15
        elif dining_total >= 2:
            score += 10
        
        # Parks/Recreation (max 20 points)
        park_count = amenities_data['parks']['count_within_1mile']
        nearest_park = amenities_data['parks']['nearest']
        
        if nearest_park:
            distance = nearest_park['distance_miles']
            if distance <= 0.5:
                score += 20
            elif distance <= 0.75:
                score += 15
            elif distance <= 1.0:
                score += 10
        
        # Transit accessibility (max 15 points)
        transit_count = amenities_data['transit']['count_within_1mile']
        nearest_transit = amenities_data['transit']['nearest']
        
        if nearest_transit:
            distance = nearest_transit['distance_miles']
            if distance <= 0.25:
                score += 15
            elif distance <= 0.5:
                score += 10
            elif distance <= 0.75:
                score += 5
        
        # Schools (max 15 points)
        school_count = amenities_data['schools']['count_within_1mile']
        if school_count >= 3:
            score += 15
        elif school_count >= 2:
            score += 10
        elif school_count >= 1:
            score += 5
        
        return min(score, 100)
    
    def _generate_summary(self, walkability_score: int, amenities_data: dict) -> str:
        """Generate human-readable summary."""
        parts = []
        
        # Overall score
        if walkability_score >= 90:
            parts.append("This property has excellent walkability.")
        elif walkability_score >= 70:
            parts.append("This property is very walkable.")
        elif walkability_score >= 50:
            parts.append("This property is somewhat walkable.")
        else:
            parts.append("This property is car-dependent.")
        
        # Highlight key amenities
        grocery_count = amenities_data['groceries']['count_within_1mile']
        if grocery_count > 0:
            parts.append(f"{grocery_count} grocery store(s) within 1 mile.")
        
        dining_total = (amenities_data['restaurants']['count_within_1mile'] + 
                       amenities_data['cafes']['count_within_1mile'])
        if dining_total > 0:
            parts.append(f"{dining_total} dining option(s) nearby.")
        
        park_count = amenities_data['parks']['count_within_1mile']
        if park_count > 0:
            parts.append(f"{park_count} park(s) for recreation.")
        
        return " ".join(parts)
    
    def __call__(
        self,
        latitude: float,
        longitude: float,
        radius_miles: float = 1.0
    ) -> WalkabilityOutput:
        """
        Assess walkability for a property location.
        
        Args:
            latitude: Property latitude
            longitude: Property longitude
            radius_miles: Search radius in miles (default 1.0)
            
        Returns:
            WalkabilityOutput with score and amenity details
        """
        logger.info(f"Assessing walkability for ({latitude}, {longitude})")
        
        try:
            # Convert radius to meters
            radius_meters = int(radius_miles * 1609.34)
            
            # Query OSM
            raw_data = self._query_osm_amenities(latitude, longitude, radius_meters)
            
            # Process amenities
            processed = self._process_amenities(raw_data, latitude, longitude)
            
            # Calculate walkability score
            walkability_score = self._calculate_walkability_score(processed)
            
            # Determine interpretation
            if walkability_score >= 90:
                interpretation = "Walker's Paradise"
            elif walkability_score >= 70:
                interpretation = "Very Walkable"
            elif walkability_score >= 50:
                interpretation = "Somewhat Walkable"
            elif walkability_score >= 25:
                interpretation = "Car-Dependent"
            else:
                interpretation = "Very Car-Dependent"
            
            # Generate summary
            summary = self._generate_summary(walkability_score, processed)
            
            # Build output
            output = WalkabilityOutput(
                walkability_score=walkability_score,
                walkability_interpretation=interpretation,
                groceries=AmenityCategoryInfo(**processed['groceries']),
                restaurants=AmenityCategoryInfo(**processed['restaurants']),
                schools=AmenityCategoryInfo(**processed['schools']),
                parks=AmenityCategoryInfo(**processed['parks']),
                hospitals=AmenityCategoryInfo(**processed['hospitals']),
                transit=AmenityCategoryInfo(**processed['transit']),
                cafes=AmenityCategoryInfo(**processed['cafes']),
                summary=summary
            )
            
            logger.info(f"Walkability score: {walkability_score} ({interpretation})")
            return output
            
        except Exception as e:
            logger.error(f"Error assessing walkability: {e}")
            raise
    
    def get_tool_schema(self) -> Dict[str, Any]:
        """Return JSON schema for MCP protocol compatibility."""
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": WalkabilityInput.model_json_schema(),
            "output_schema": WalkabilityOutput.model_json_schema()
        }

In [18]:
# Initialize walkability tool
walkability_tool = WalkabilityTool()

INFO:__main__:WalkabilityTool initialized


In [19]:
# Example 1: Downtown Orlando
result = walkability_tool(
    latitude=28.5383,
    longitude=-81.3792,
    radius_miles=1.0
)

print(f"Walkability Score: {result.walkability_score}/100")
print(f"Interpretation: {result.walkability_interpretation}")
print(f"\nSummary: {result.summary}")
print(f"\nGrocery Stores: {result.groceries.count_within_1mile} within 1 mile")
if result.groceries.nearest:
    print(f"  Nearest: {result.groceries.nearest.name} ({result.groceries.nearest.distance_miles} mi)")

INFO:__main__:Assessing walkability for (28.5383, -81.3792)
ERROR:__main__:OSM API error: 504 Server Error: Gateway Timeout for url: http://overpass-api.de/api/interpreter
ERROR:__main__:Error assessing walkability: 504 Server Error: Gateway Timeout for url: http://overpass-api.de/api/interpreter


HTTPError: 504 Server Error: Gateway Timeout for url: http://overpass-api.de/api/interpreter

In [ ]:
# Example 2: Suburban location
result2 = walkability_tool(
    latitude=28.4500,
    longitude=-81.3800
)

print(f"Walkability Score: {result2.walkability_score}/100")
print(f"Interpretation: {result2.walkability_interpretation}")
print(f"\nSummary: {result2.summary}")
print(f"\nGrocery Stores: {result2.groceries.count_within_1mile} within 1 mile")
if result2.groceries.nearest:
    print(f"  Nearest: {result2.groceries.nearest.name} ({result2.groceries.nearest.distance_miles} mi)")

# Market Expert

In [20]:
# ============================================================================
# PYDANTIC SCHEMAS
# ============================================================================

class MarketExpertInput(BaseModel):
    """Input schema for Orlando market expert queries"""
    query: str = Field(
        ..., 
        description="Question about Orlando real estate market",
        examples=[
            "What are the best neighborhoods for families in Orlando?",
            "How is the Orlando housing market trending?",
            "What should I know about buying in Winter Park?"
        ]
    )
    temperature: float = Field(
        default=0.7,
        description="Response creativity (0=focused, 1=creative)",
        ge=0.0,
        le=1.0
    )
    max_tokens: int = Field(
        default=500,
        description="Maximum response length",
        ge=50,
        le=2000
    )


class MarketExpertOutput(BaseModel):
    """Output schema for Orlando market expert responses"""
    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "query": "What are the best neighborhoods for families?",
                "response": "Based on Orlando market data, the top family-friendly neighborhoods are...",
                "model_used": "ft:gpt-4o-mini-2024-07-18:personal::D9f85khV",
                "tokens_used": 245
            }
        }
    )
    
    query: str = Field(..., description="Original query")
    response: str = Field(..., description="Expert response from fine-tuned model")
    model_used: str = Field(..., description="Fine-tuned model ID")
    tokens_used: int = Field(..., description="Total tokens consumed")


class FineTunedModelConfig(BaseModel):
    """Configuration for the fine-tuned model"""
    model_id: str = Field(..., description="OpenAI fine-tuned model ID")
    system_prompt: str = Field(..., description="System prompt for the model")
    project: str = Field(..., description="Project name/description")
    date_trained: str = Field(..., description="Date model was trained")


# ============================================================================
# MARKET EXPERT TOOL CLASS
# ============================================================================

class MarketExpertTool:
    """
    MCP-compatible tool for querying fine-tuned Orlando real estate market expert.
    
    This tool uses a fine-tuned GPT-4o-mini model trained on Orlando-specific
    market data, trends, neighborhoods, and local real estate insights.
    """
    
    name: str = "orlando_market_expert"
    description: str = """
    Query a fine-tuned AI expert on Orlando real estate market insights.
    
    This expert has been trained on Orlando-specific data including:
    - Neighborhood characteristics and trends
    - Local market conditions and pricing
    - School districts and amenities
    - Investment opportunities
    - Buyer/seller considerations
    - Historical market patterns
    
    Use this tool for:
    - Market analysis and trends
    - Neighborhood recommendations
    - Investment advice
    - Local market knowledge
    - Comparative neighborhood analysis
    
    The model provides contextual, data-driven insights specific to Orlando, FL.
    """
    
    def __init__(
        self,
        model_config_path: str = "../output/orlando_mkt_data_expert_metadata.json",
        openai_api_key: Optional[str] = None
    ):
        """
        Initialize Market Expert Tool with fine-tuned model.
        
        Args:
            model_config_path: Path to JSON config file with model details
            openai_api_key: OpenAI API key (reads from env if not provided)
        """
        # Get API key
        self.openai_api_key = openai_api_key or os.getenv("OPENAI_API_KEY")
        if not self.openai_api_key:
            raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY environment variable.")
        
        # Initialize OpenAI client
        self.openai_client = OpenAI(api_key=self.openai_api_key)
        
        # Load model configuration
        logger.info(f"Loading fine-tuned model config from {model_config_path}")
        with open(model_config_path, 'r') as f:
            config_data = json.load(f)
        
        self.model_config = FineTunedModelConfig(**config_data)
        
        logger.info(
            f"MarketExpertTool initialized with model: {self.model_config.model_id}\n"
            f"Project: {self.model_config.project}\n"
            f"Trained: {self.model_config.date_trained}"
        )
    
    def __call__(
        self,
        query: str,
        temperature: float = 0.7,
        max_tokens: int = 500
    ) -> MarketExpertOutput:
        """
        Query the Orlando market expert.
        
        Args:
            query: Question about Orlando real estate market
            temperature: Response creativity (0=focused, 1=creative)
            max_tokens: Maximum response length
            
        Returns:
            MarketExpertOutput with expert response
        """
        logger.info(f"Querying Orlando market expert: '{query[:50]}...'")
        
        try:
            # Call fine-tuned model
            response = self.openai_client.chat.completions.create(
                model=self.model_config.model_id,
                messages=[
                    {
                        "role": "system",
                        "content": self.model_config.system_prompt
                    },
                    {
                        "role": "user",
                        "content": query
                    }
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            
            # Extract response
            expert_response = response.choices[0].message.content
            tokens_used = response.usage.total_tokens
            
            # Build output
            output = MarketExpertOutput(
                query=query,
                response=expert_response,
                model_used=self.model_config.model_id,
                tokens_used=tokens_used
            )
            
            logger.info(f"Market expert response generated ({tokens_used} tokens)")
            return output
            
        except Exception as e:
            logger.error(f"Error querying market expert: {e}")
            raise
    
    def get_tool_schema(self) -> Dict[str, Any]:
        """Return JSON schema for MCP protocol compatibility."""
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": MarketExpertInput.model_json_schema(),
            "output_schema": MarketExpertOutput.model_json_schema()
        }


# ============================================================================
# HELPER: Create Model Config File
# ============================================================================

def create_model_config_file(
    model_id: str,
    system_prompt: str,
    project: str,
    date_trained: str,
    output_path: str = "../output/orlando_mkt_data_expert_metadata.json"
):
    """
    Helper function to create the model config JSON file.
    
    Args:
        model_id: Fine-tuned model ID from OpenAI
        system_prompt: System prompt for the model
        project: Project name/description
        date_trained: Training date (YYYY-MM-DD format)
        output_path: Where to save the config file
    """
    config = {
        "model_id": model_id,
        "system_prompt": system_prompt,
        "project": project,
        "date_trained": date_trained
    }
    
    # Ensure output directory exists
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    
    # Save config
    with open(output_path, 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"Model config saved to: {output_path}")

In [ ]:
# Run this ONCE to create orlando_mkt_data_expert_metadata.json
create_model_config_file(
    model_id="ft:gpt-4o-mini-2024-07-18:personal::D9f85khV",
    system_prompt="You are an Orlando real estate expert.",
    project="In-House Orlando Market Data Expert",
    date_trained="2026-02-15",
    output_path="../output/orlando_mkt_data_expert_metadata.json"
)

Model config saved to: ../output/orlando_mkt_data_expert_metadata.json


In [23]:
# Initialize the market expert tool
market_expert = MarketExpertTool(
    model_config_path="../output/orlando_mkt_data_expert_metadata.json"
)

INFO:__main__:Loading fine-tuned model config from ../output/orlando_mkt_data_expert_metadata.json


INFO:__main__:MarketExpertTool initialized with model: ft:gpt-4o-mini-2024-07-18:personal::D9f85khV
Project: In-House Orlando Market Data Expert
Trained: 2026-02-15


In [24]:
def print_expert_response(result, title="Orlando Market Expert"):
    """Print expert response in a formatted box"""
    print("\n" + "="*80)
    print(f"  {title}")
    print("="*80)
    print(f"\n📍 Query: {result.query}\n")
    print("💬 Expert Response:")
    print("-"*80)
    print(result.response)
    print("-"*80)
    print(f"🤖 Model: {result.model_used.split('::')[-1]}")
    print(f"📊 Tokens Used: {result.tokens_used}")
    print("="*80 + "\n")

In [ ]:
# Example 1: Ask about neighborhoods

result = market_expert(
    query="What are the best neighborhoods for families with young children in Orlando?"
)
print_expert_response(result, "Market Trends Analysis")

In [ ]:
# Example 2: Market trends
result2 = market_expert(
    query="How is the Orlando housing market trending right now? Should I wait to buy?",
    temperature=0.5  # More focused response
)
print_expert_response(result2, "Market Trends Analysis")


# Example 3: Investment advice
result3 = market_expert(
    query="I'm looking to invest in rental properties in Orlando. Which areas have the best ROI?",
    temperature=0.7,
    max_tokens=800  # Longer response
)
print_expert_response(result3, "Market Trends Analysis")


# Example 4: Neighborhood comparison
result4 = market_expert(
    query="Compare Winter Park vs Lake Nona for young professionals"
)
print_expert_response(result4, "Market Trends Analysis")


# Example 5: School districts
result5 = market_expert(
    query="Which Orlando neighborhoods have the best public schools?"
)
print_expert_response(result5, "Market Trends Analysis")

# Summary

In [25]:
# ============================================================================
# SUMMARY: ALL TOOLS READY
# ============================================================================


print("="*80)
print("  MCP LAYER - TOOL INVENTORY")
print("="*80)

tools = [
    ("zoning_law_query", ZoningLawTool, "Query Orlando zoning laws"),
    ("property_damage_assessment", VisionTool, "Assess property damage from images"),
    ("predict_fair_market_value", FMVTool, "Predict property FMV"),
    ("assess_walkability", WalkabilityTool, "Calculate walkability score"),
    ("orlando_market_expert", MarketExpertTool, "Query market expert")
]

for name, tool_class, description in tools:
    print(f"\n✅ {name}")
    print(f"   Class: {tool_class.__name__}")
    print(f"   Description: {description}")

print("\n" + "="*80)
print("  ALL TOOLS READY FOR AGENT INTEGRATION")
print("="*80)

  MCP LAYER - TOOL INVENTORY

✅ zoning_law_query
   Class: ZoningLawTool
   Description: Query Orlando zoning laws

✅ property_damage_assessment
   Class: VisionTool
   Description: Assess property damage from images

✅ predict_fair_market_value
   Class: FMVTool
   Description: Predict property FMV

✅ assess_walkability
   Class: WalkabilityTool
   Description: Calculate walkability score

✅ orlando_market_expert
   Class: MarketExpertTool
   Description: Query market expert

  ALL TOOLS READY FOR AGENT INTEGRATION


In [26]:
def validate_all_tools():
    """Quick validation that all tools can be initialized"""
    try:
        print("Validating tools...")
        
        # Test each tool initialization
        z = ZoningLawTool(index_name="orlando-zoning-index", namespace="__default__")
        print("✅ ZoningLawTool")
        
        v = VisionTool()
        print("✅ VisionTool")
        
        f = FMVTool()
        print("✅ FMVTool")
        
        w = WalkabilityTool()
        print("✅ WalkabilityTool")
        
        m = MarketExpertTool()
        print("✅ MarketExpertTool")
        
        print("\n🎉 All tools validated successfully!")
        return True
        
    except Exception as e:
        print(f"\n❌ Validation failed: {e}")
        return False

# Run validation
validate_all_tools()

INFO:__main__:Initializing Pinecone client...
INFO:__main__:Initializing OpenAI client...
INFO:__main__:ZoningLawTool initialized successfully with index 'orlando-zoning-index'
INFO:__main__:Initializing OpenAI client for vision analysis...
INFO:__main__:Initializing Pinecone client...
INFO:__main__:VisionTool initialized successfully
INFO:__main__:FMVTool initialized with 30 features
INFO:__main__:WalkabilityTool initialized
INFO:__main__:Loading fine-tuned model config from ../output/orlando_mkt_data_expert_metadata.json
INFO:__main__:MarketExpertTool initialized with model: ft:gpt-4o-mini-2024-07-18:personal::D9f85khV
Project: In-House Orlando Market Data Expert
Trained: 2026-02-15


Validating tools...
✅ ZoningLawTool
✅ VisionTool
✅ FMVTool
✅ WalkabilityTool
✅ MarketExpertTool

🎉 All tools validated successfully!


True